# 03 — Generate Mutation Panel

In this step, I generate a small, biologically interpretable mutation panel for lysozyme (1LYZ).

Selected residues were chosen from:
- buried hydrophobic core positions
- exposed surface polar/charged positions

For each selected residue, I define:
- a conservative mutation
- a moderate mutation
- a disruptive mutation

This creates a controlled set of mutations for stability-focused analysis.

In [12]:
import pandas as pd
from pathlib import Path

df = pd.read_csv("../data/processed/residue_features.csv")
df.head()

,chain,position,insertion_code,resname_3,resname_1,aa_dssp,secondary_structure,ss_simple,rsa,environment
0,A,1,NaN,LYS,K,K,-,loop,0.473171,surface
1,A,2,NaN,VAL,V,V,B,loop,0.626761,surface
2,A,3,NaN,PHE,F,F,-,loop,0.096447,core
3,A,4,NaN,GLY,G,G,-,loop,0.345238,surface
4,A,5,NaN,ARG,R,R,H,helix,0.314516,surface


***Look at core residues***

In [2]:
core_df = df[df["environment"] == "core"]
core_df[["position", "resname_1", "ss_simple"]].head(20)

,position,resname_1,ss_simple
2,3,F,loop
7,8,L,helix
8,9,A,helix
11,12,M,helix
16,17,L,loop
24,25,L,loop
25,26,G,helix
27,28,W,helix
28,29,V,helix
29,30,C,helix


***Filter good core candidates***

In [10]:
core_candidates = core_df[
    core_df["resname_1"].isin(["L", "I", "V", "F", "W", "M"])
]

core_candidates.head(50)

,chain,position,insertion_code,resname_3,resname_1,aa_dssp,secondary_structure,ss_simple,rsa,environment
2,A,3,NaN,PHE,F,F,-,loop,0.096447,core
7,A,8,NaN,LEU,L,L,H,helix,0.000000,core
11,A,12,NaN,MET,M,M,H,helix,0.015957,core
16,A,17,NaN,LEU,L,L,-,loop,0.000000,core
24,A,25,NaN,LEU,L,L,T,loop,0.012195,core
27,A,28,NaN,TRP,W,W,H,helix,0.000000,core
28,A,29,NaN,VAL,V,V,H,helix,0.000000,core
54,A,55,NaN,ILE,I,I,T,loop,0.017751,core
55,A,56,NaN,LEU,L,L,T,loop,0.000000,core
57,A,58,NaN,ILE,I,I,E,sheet,0.017751,core


***surface candidate table***

In [11]:
surface_df = df[df["environment"] == "surface"].copy()

surface_candidates = surface_df[
    surface_df["resname_1"].isin(["D", "E", "K", "R", "N", "Q", "S", "T"])
].copy()

surface_candidates = surface_candidates.sort_values("position")

surface_candidates[["position", "resname_1", "ss_simple", "rsa"]]

,position,resname_1,ss_simple,rsa
0,1,K,loop,0.473171
4,5,R,helix,0.314516
6,7,E,helix,0.463918
12,13,K,helix,0.331707
13,14,R,helix,0.810484
17,18,D,loop,0.343558
18,19,N,loop,0.630573
20,21,R,loop,0.612903
32,33,K,helix,0.273171
36,37,N,loop,0.585987


***Define selected residues***

In [13]:
selected_sites = [
    {"position": 8,  "wt": "L", "environment_group": "core"},
    {"position": 29, "wt": "V", "environment_group": "core"},
    {"position": 56, "wt": "L", "environment_group": "core"},
    {"position": 18, "wt": "D", "environment_group": "surface"},
    {"position": 73, "wt": "R", "environment_group": "surface"},
    {"position": 41, "wt": "Q", "environment_group": "surface"},
]

selected_df = pd.DataFrame(selected_sites)
selected_df

,position,wt,environment_group
0,8,L,core
1,29,V,core
2,56,L,core
3,18,D,surface
4,73,R,surface
5,41,Q,surface


***Define selected residues***

In [14]:
mutation_plan = {
    ("L", "core"): [
        {"mut": "I", "mutation_class": "conservative", "rationale": "Maintains hydrophobic character with similar side-chain size and packing behavior."},
        {"mut": "A", "mutation_class": "moderate",     "rationale": "Reduces side-chain volume and may create a cavity in the hydrophobic core."},
        {"mut": "D", "mutation_class": "disruptive",   "rationale": "Introduces a charged residue into the buried core, which is often destabilizing."},
    ],
    ("V", "core"): [
        {"mut": "I", "mutation_class": "conservative", "rationale": "Retains hydrophobic chemistry with a similar branched side chain."},
        {"mut": "A", "mutation_class": "moderate",     "rationale": "Shrinks side-chain volume and may weaken tight core packing."},
        {"mut": "D", "mutation_class": "disruptive",   "rationale": "Introduces a charged residue into a buried hydrophobic environment."},
    ],
    ("D", "surface"): [
        {"mut": "E", "mutation_class": "conservative", "rationale": "Retains negative charge while slightly extending side-chain length."},
        {"mut": "N", "mutation_class": "moderate",     "rationale": "Removes formal charge but preserves polar hydrogen-bonding potential."},
        {"mut": "F", "mutation_class": "disruptive",   "rationale": "Introduces a hydrophobic aromatic residue at an exposed polar site."},
    ],
    ("R", "surface"): [
        {"mut": "K", "mutation_class": "conservative", "rationale": "Retains positive charge with similar surface electrostatic role."},
        {"mut": "Q", "mutation_class": "moderate",     "rationale": "Removes positive charge while keeping a polar side chain."},
        {"mut": "F", "mutation_class": "disruptive",   "rationale": "Introduces a bulky hydrophobic aromatic residue at a charged surface position."},
    ],
    ("Q", "surface"): [
        {"mut": "N", "mutation_class": "conservative", "rationale": "Retains polar amide chemistry with a slightly shorter side chain."},
        {"mut": "E", "mutation_class": "moderate",     "rationale": "Introduces a negative charge at a previously neutral polar site."},
        {"mut": "F", "mutation_class": "disruptive",   "rationale": "Replaces a polar surface residue with a hydrophobic aromatic side chain."},
    ],
}

***Generate mutation rows***

In [15]:
mutation_rows = []

for _, row in selected_df.iterrows():
    wt = row["wt"]
    position = row["position"]
    environment_group = row["environment_group"]

    plans = mutation_plan[(wt, environment_group)]

    for plan in plans:
        mut = plan["mut"]
        mutation_id = f"{wt}{position}{mut}"

        mutation_rows.append({
            "mutation_id": mutation_id,
            "position": position,
            "wt": wt,
            "mut": mut,
            "environment_group": environment_group,
            "mutation_class": plan["mutation_class"],
            "rationale": plan["rationale"]
        })

mut_df = pd.DataFrame(mutation_rows)
mut_df

,mutation_id,position,wt,mut,environment_group,mutation_class,rationale
0,L8I,8,L,I,core,conservative,Maintains hydrophobic character with similar s...
1,L8A,8,L,A,core,moderate,Reduces side-chain volume and may create a cav...
2,L8D,8,L,D,core,disruptive,Introduces a charged residue into the buried c...
3,V29I,29,V,I,core,conservative,Retains hydrophobic chemistry with a similar b...
4,V29A,29,V,A,core,moderate,Shrinks side-chain volume and may weaken tight...
5,V29D,29,V,D,core,disruptive,Introduces a charged residue into a buried hyd...
6,L56I,56,L,I,core,conservative,Maintains hydrophobic character with similar s...
7,L56A,56,L,A,core,moderate,Reduces side-chain volume and may create a cav...
8,L56D,56,L,D,core,disruptive,Introduces a charged residue into the buried c...
9,D18E,18,D,E,surface,conservative,Retains negative charge while slightly extendi...


***Merge mutation table with residue features***

In [16]:
mutation_df = mut_df.merge(
    df[["position", "resname_3", "resname_1", "ss_simple", "rsa", "environment"]],
    on="position",
    how="left"
)

mutation_df

,mutation_id,position,wt,mut,environment_group,mutation_class,rationale,resname_3,resname_1,ss_simple,rsa,environment
0,L8I,8,L,I,core,conservative,Maintains hydrophobic character with similar s...,LEU,L,helix,0.000000,core
1,L8A,8,L,A,core,moderate,Reduces side-chain volume and may create a cav...,LEU,L,helix,0.000000,core
2,L8D,8,L,D,core,disruptive,Introduces a charged residue into the buried c...,LEU,L,helix,0.000000,core
3,V29I,29,V,I,core,conservative,Retains hydrophobic chemistry with a similar b...,VAL,V,helix,0.000000,core
4,V29A,29,V,A,core,moderate,Shrinks side-chain volume and may weaken tight...,VAL,V,helix,0.000000,core
5,V29D,29,V,D,core,disruptive,Introduces a charged residue into a buried hyd...,VAL,V,helix,0.000000,core
6,L56I,56,L,I,core,conservative,Maintains hydrophobic character with similar s...,LEU,L,loop,0.000000,core
7,L56A,56,L,A,core,moderate,Reduces side-chain volume and may create a cav...,LEU,L,loop,0.000000,core
8,L56D,56,L,D,core,disruptive,Introduces a charged residue into the buried c...,LEU,L,loop,0.000000,core
9,D18E,18,D,E,surface,conservative,Retains negative charge while slightly extendi...,ASP,D,loop,0.343558,surface


***Reorder columns for readability***

In [17]:
mutation_df = mutation_df[
    [
        "mutation_id",
        "position",
        "resname_3",
        "resname_1",
        "wt",
        "mut",
        "environment_group",
        "environment",
        "ss_simple",
        "rsa",
        "mutation_class",
        "rationale"
    ]
]

mutation_df

,mutation_id,position,resname_3,resname_1,wt,mut,environment_group,environment,ss_simple,rsa,mutation_class,rationale
0,L8I,8,LEU,L,L,I,core,core,helix,0.000000,conservative,Maintains hydrophobic character with similar s...
1,L8A,8,LEU,L,L,A,core,core,helix,0.000000,moderate,Reduces side-chain volume and may create a cav...
2,L8D,8,LEU,L,L,D,core,core,helix,0.000000,disruptive,Introduces a charged residue into the buried c...
3,V29I,29,VAL,V,V,I,core,core,helix,0.000000,conservative,Retains hydrophobic chemistry with a similar b...
4,V29A,29,VAL,V,V,A,core,core,helix,0.000000,moderate,Shrinks side-chain volume and may weaken tight...
5,V29D,29,VAL,V,V,D,core,core,helix,0.000000,disruptive,Introduces a charged residue into a buried hyd...
6,L56I,56,LEU,L,L,I,core,core,loop,0.000000,conservative,Maintains hydrophobic character with similar s...
7,L56A,56,LEU,L,L,A,core,core,loop,0.000000,moderate,Reduces side-chain volume and may create a cav...
8,L56D,56,LEU,L,L,D,core,core,loop,0.000000,disruptive,Introduces a charged residue into the buried c...
9,D18E,18,ASP,D,D,E,surface,surface,loop,0.343558,conservative,Retains negative charge while slightly extendi...
